
# Lab 5 — Train Your YOLOv5 Dish Model (Box + Seg)

**ผู้สอน:** rattachai  
**ส่ง:** พรุ่งนี้ 10:00 น. (Asia/Bangkok)

โน้ตบุ๊กนี้เตรียมโค้ดครบถ้วนสำหรับ:
- เทรน **YOLOv5 (box-detection)** จาก `yolov5s.pt`
- เทรน **YOLOv5-seg (instance segmentation)** จาก `yolov5s-seg.pt`
- แสดง **TensorBoard**: box regression loss, objectness loss, classification loss, segmentation loss, learning-rate, precision/recall/mAP, PR Curve และ mAP@0.5
- รวมความสามารถ **bounding box + segmentation mask** เพื่อทำ **instance segmentation overlay**
- สร้าง **รูปตัวอย่างผลลัพธ์อย่างน้อย 4 รูป** พร้อม Overlay (boxes + masks) และคำอธิบายใต้ภาพ

> ⚠️ ก่อนรัน: ปรับพาธของชุดข้อมูลและชื่อคลาสให้ถูกต้องในเซลล์ **Config** ด้านล่าง


In [ ]:

# === Config (กรุณาปรับให้ตรงกับเครื่องคุณ) ===
DATA_ROOT = r"/path/to/your/dataset"  # โฟลเดอร์หลักของชุดข้อมูล (มี images/train, images/val, labels/...)
DATA_YAML = r"/path/to/your/data.yaml"  # ไฟล์ data.yaml รูปแบบ YOLOv5 (names, train, val)
CLASS_NAMES = ["dish"]  # รายชื่อคลาสตามจริง เช่น ["plate","spoon","fork","rice","curry"]

# Hyperparameters ที่ต้องแสดงการลู่เข้า: ปรับให้เหมาะกับ GPU/เวลา
EPOCHS_BOX = 50
BATCH_BOX = 16

EPOCHS_SEG = 50
BATCH_SEG = 8

# ขนาดภาพ (ปรับตาม dataset)
IMG_SIZE = 640

# จำนวนตัวอย่างรูปผลลัพธ์ที่ต้องการเซฟ
N_SAMPLES = 4

# เลือกอุปกรณ์: "0" = GPU0, "cpu" = รันบน CPU (ช้ากว่า)
DEVICE = "0"


## 1) Environment Setup

In [ ]:

# ติดตั้งไลบรารีหลักที่ใช้ (รันครั้งแรกเท่านั้น)
# ถ้าใช้งานใน Colab ให้ uncomment บรรทัด pip และ git clone
# !pip install --upgrade pip
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install tensorboard opencv-python matplotlib pandas seaborn scikit-learn PyYAML
# !sudo apt-get install -y libgl1-mesa-glx  # บางเครื่องต้องใช้เพื่อให้ cv2.imshow ได้

# ดาวน์โหลด YOLOv5 repo (อัปเดตรุ่นที่รองรับ segmentation)
# !git clone https://github.com/ultralytics/yolov5.git
# %cd yolov5
# !pip install -r requirements.txt

import os, sys, shutil, json, math, glob
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ตรวจสอบโครงสร้างโฟลเดอร์เบื้องต้น
print("DATA_ROOT =", DATA_ROOT)
print("DATA_YAML =", DATA_YAML)
print("Classes   =", CLASS_NAMES)


### ตรวจสอบไฟล์ `data.yaml`

In [ ]:

from pathlib import Path
import yaml

assert Path(DATA_YAML).exists(), f"ไม่พบไฟล์ data.yaml: {DATA_YAML}"
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("data.yaml content:")
print(json.dumps(data_cfg, indent=2, ensure_ascii=False))


## 2) TensorBoard (สำหรับดูการลู่เข้า)

In [ ]:

# เริ่ม TensorBoard (ถ้ารันใน Jupyter/Colab)
# from torch.utils.tensorboard import SummaryWriter
# %load_ext tensorboard
# %tensorboard --logdir runs --reload_interval 5
print("หลังเทรน จะสามารถเปิด TensorBoard ด้วยคำสั่ง:")
print("  tensorboard --logdir runs")


## 3) Train YOLOv5 (Box Detection)

In [ ]:

# รันการเทรน YOLOv5 (Box) — ต้องอยู่ในโฟลเดอร์ yolov5
# ถ้าอยู่โฟลเดอร์อื่นให้เปลี่ยนเป็นพาธเต็มของ train.py
# ตัวอย่าง (เมื่ออยู่ในโฟลเดอร์ yolov5):
# !python train.py --img $IMG_SIZE --batch $BATCH_BOX --epochs $EPOCHS_BOX --data $DATA_YAML \
#     --weights yolov5s.pt --project runs/train_box --name exp --exist-ok --device $DEVICE

print("คำสั่งตัวอย่าง (โปรดรันเองเมื่อพร้อม):")
print(f"python yolov5/train.py --img {IMG_SIZE} --batch {BATCH_BOX} --epochs {EPOCHS_BOX} "
      f"--data {DATA_YAML} --weights yolov5s.pt --project runs/train_box --name exp "
      f"--exist-ok --device {DEVICE}")


## 4) Train YOLOv5-seg (Instance Segmentation)

In [ ]:

# รันการเทรน YOLOv5-seg — ต้องอยู่ในโฟลเดอร์ yolov5/segment
# ตัวอย่าง:
# !python segment/train.py --img $IMG_SIZE --batch $BATCH_SEG --epochs $EPOCHS_SEG --data $DATA_YAML \
#     --weights yolov5s-seg.pt --project runs/train_seg --name exp --exist-ok --device $DEVICE

print("คำสั่งตัวอย่าง (โปรดรันเองเมื่อพร้อม):")
print(f"python yolov5/segment/train.py --img {IMG_SIZE} --batch {BATCH_SEG} --epochs {EPOCHS_SEG} "
      f"--data {DATA_YAML} --weights yolov5s-seg.pt --project runs/train_seg --name exp "
      f"--exist-ok --device {DEVICE}")


## 5) Evaluate + Export PR Curve & mAP@0.5

In [ ]:

# หลังเทรนเสร็จ ให้ประเมินโมเดลและสร้าง PR curve/mAP
# กล่อง: yolov5/val.py, เซกเมนต์: yolov5/segment/val.py
# ตัวอย่าง:
# !python yolov5/val.py --weights runs/train_box/exp/weights/best.pt --data $DATA_YAML --img $IMG_SIZE --task val --device $DEVICE --save-json --verbose
# !python yolov5/segment/val.py --weights runs/train_seg/exp/weights/best.pt --data $DATA_YAML --img $IMG_SIZE --task val --device $DEVICE --save-json --verbose

print("หลังรันวัดผล ดูไฟล์ใน runs/val/* เช่น PR_curve.png, results.csv, labels_correlogram.png")


## 6) แสดงกราฟการลู่เข้า (จาก `results.csv`)

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def plot_results_csv(csv_path, title):
    if not Path(csv_path).exists():
        print(f"ไม่พบไฟล์: {csv_path}")
        return
    df = pd.read_csv(csv_path)
    print(df.head())

    # ชื่อคอลัมน์ใน YOLOv5 results.csv อาจรวมถึง: box_loss, obj_loss, cls_loss, dfl_loss, seg_loss, lr0, lrf, ... , mAP_0.5
    # จะลองหาและ plot เฉพาะที่มีอยู่
    cols_map = {
        "box": ["box_loss", "loss_box", "boxes_loss"],
        "obj": ["obj_loss", "loss_obj"],
        "cls": ["cls_loss", "loss_cls"],
        "seg": ["seg_loss", "mask_loss", "loss_seg"],
        "lr":  ["lr0", "lr1", "lr2"],
        "map50": ["mAP_0.5", "metrics/mAP_0.5"]
    }

    def find_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    box_c = find_col(df, cols_map["box"])
    obj_c = find_col(df, cols_map["obj"])
    cls_c = find_col(df, cols_map["cls"])
    seg_c = find_col(df, cols_map["seg"])
    map_c = find_col(df, cols_map["map50"])

    # Losses
    for label, col in [("Box regression-loss", box_c),
                       ("Objectness-loss", obj_c),
                       ("Classification-loss", cls_c),
                       ("Segmentation-loss", seg_c)]:
        if col:
            plt.figure()
            plt.plot(df[col])
            plt.title(f"{title}: {label}")
            plt.xlabel("Epoch")
            plt.ylabel(label)
            plt.grid(True)
            plt.show()
        else:
            print(f"ไม่พบคอลัมน์สำหรับ {label}")

    # Learning rate
    lr_cols = [c for c in cols_map["lr"] if c in df.columns]
    if lr_cols:
        plt.figure()
        for c in lr_cols:
            plt.plot(df[c], label=c)
        plt.title(f"{title}: Learning-rate")
        plt.xlabel("Epoch")
        plt.ylabel("LR")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("ไม่พบคอลัมน์ Learning-rate")

    # Accuracy Plot (ใช้ mAP@0.5 แทน)
    if map_c:
        plt.figure()
        plt.plot(df[map_c])
        plt.title(f"{title}: Accuracy (mAP@0.5)")
        plt.xlabel("Epoch")
        plt.ylabel("mAP@0.5")
        plt.grid(True)
        plt.show()
    else:
        print("ไม่พบคอลัมน์ mAP@0.5 ใน results.csv")

# ตัวอย่าง path (ปรับตามจริง)
plot_results_csv("runs/train_box/exp/results.csv", "YOLOv5 Box")
plot_results_csv("runs/train_seg/exp/results.csv", "YOLOv5 Seg")


### (Optional) บันทึกกราฟลง TensorBoard แบบกำหนดเอง

In [ ]:

# ถ้าต้องการบันทึกกราฟลง TensorBoard เองจาก results.csv
# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter(log_dir="runs/tb_custom")

# df_box = pd.read_csv("runs/train_box/exp/results.csv") if Path("runs/train_box/exp/results.csv").exists() else None
# if df_box is not None:
#     for i, row in df_box.iterrows():
#         for k, v in row.items():
#             if isinstance(v, (int,float)) and not np.isnan(v):
#                 writer.add_scalar(f"box/{k}", v, i)

# df_seg = pd.read_csv("runs/train_seg/exp/results.csv") if Path("runs/train_seg/exp/results.csv").exists() else None
# if df_seg is not None:
#     for i, row in df_seg.iterrows():
#         for k, v in row.items():
#             if isinstance(v, (int,float)) and not np.isnan(v):
#                 writer.add_scalar(f"seg/{k}", v, i)

# writer.flush()
# writer.close()
print("ตัวอย่างโค้ดสำหรับบันทึก TensorBoard แบบกำหนดเอง (ถูกคอมเมนต์ไว้)")


## 7) Inference + Instance Segmentation Overlay (Boxes + Masks)

In [ ]:

# สคริปต์นี้จะ:
# 1) ใช้โมเดล YOLOv5-seg (best.pt) ทำ inference กับรูปจาก val set
# 2) Overlay ทั้ง bounding boxes และ segmentation masks ต่อ instance
# 3) เซฟรูปผลลัพธ์อย่างน้อย N_SAMPLES รูป

from pathlib import Path
import random

# พาธโมเดลที่เทรน (ปรับชื่อ exp หากต่างจากนี้)
BEST_SEG = Path("runs/train_seg/exp/weights/best.pt")
VAL_IMAGES_DIR = Path(data_cfg['val']) if isinstance(data_cfg.get('val'), str) else None

assert BEST_SEG.exists(), f"ไม่พบโมเดล: {BEST_SEG} (โปรดเทรน seg ก่อน)"
assert VAL_IMAGES_DIR and Path(VAL_IMAGES_DIR).exists(), "ไม่พบโฟลเดอร์ val images จาก data.yaml"

# ฟังก์ชัน overlay
def overlay_instance(image, masks, boxes, classes, scores, alpha=0.4):
    overlay = image.copy()
    out = image.copy()
    h, w = image.shape[:2]

    # วาด masks
    for i in range(len(masks)):
        mask = masks[i].astype(bool)
        # สร้างสีแบบสุ่มคงที่ต่อ instance
        rng = np.random.default_rng(seed=int(scores[i]*1e6)%1000000)
        color = rng.integers(low=0, high=255, size=3, dtype=np.uint8).tolist()
        overlay[mask] = (overlay[mask] * (1-alpha) + np.array(color) * alpha).astype(np.uint8)

    # ผสม overlay กับรูปเดิม
    out = cv2.addWeighted(overlay, 1.0, image, 0.0, 0)

    # วาดกรอบและฉลาก
    for i, (x1,y1,x2,y2) in enumerate(boxes):
        cls_id = int(classes[i])
        label = f"{CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else cls_id}:{scores[i]:.2f}"
        cv2.rectangle(out, (int(x1),int(y1)), (int(x2),int(y2)), (0,255,0), 2)
        cv2.putText(out, label, (int(x1), max(0,int(y1)-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2, cv2.LINE_AA)
    return out

# ใช้สคริปต์ predict ของ YOLOv5-seg เพื่อดึงผลลัพธ์ดิบ
# ตัวอย่างเรียกใช้งานตรง (โปรดรันจริงเมื่อพร้อม):
# !python yolov5/segment/predict.py --weights $BEST_SEG --source $VAL_IMAGES_DIR --save-txt --save-conf --save-csv --name seg_vis --project runs/seg_vis --img $IMG_SIZE --device $DEVICE

print("คำสั่งตัวอย่างสำหรับรันสคริปต์ predict (จะเซฟผลลัพธ์ลงโฟลเดอร์ runs/seg_vis):")
print(f"python yolov5/segment/predict.py --weights {BEST_SEG} --source {VAL_IMAGES_DIR} "
      f"--save-txt --save-conf --save-csv --name seg_vis --project runs/seg_vis --img {IMG_SIZE} --device {DEVICE}")

# ตัวอย่างการโหลดผลลัพธ์ (หากรัน predict ไปแล้ว และมีไฟล์ .npy หรือ labels ที่บันทึกไว้)


### สร้างรูป Thumbnail อย่างน้อย 4 รูป พร้อมคำอธิบายใต้ภาพ

In [ ]:

# สมมติว่าเรามีผลลัพธ์จากการรัน predict แล้วในโฟลเดอร์ runs/seg_vis/exp
# ที่ประกอบด้วยรูปภาพที่เซฟแล้ว (พร้อม overlay) หรืออย่างน้อยไฟล์ภาพต้นฉบับที่เราจะ overlay เอง
OUT_DIR = Path("runs/seg_report")
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_imgs = list(Path("runs/seg_vis/exp").glob("*.jpg")) + list(Path("runs/seg_vis/exp").glob("*.png"))
if len(candidate_imgs) < N_SAMPLES:
    print("คำเตือน: พบรูปน้อยกว่าที่ต้องการ — โปรดตรวจสอบว่าได้รัน predict แล้ว")
sample_imgs = candidate_imgs[:N_SAMPLES]

thumb_paths = []
for i, p in enumerate(sample_imgs):
    img = cv2.imread(str(p))
    if img is None:
        print("อ่านรูปไม่สำเร็จ:", p)
        continue
    # ถ้าภาพนี้เป็นภาพที่ YOLOv5 เซฟ overlay แล้ว ก็ใช้ได้เลย
    # หากเป็นรูปดิบ ต้องโหลดผลลัพธ์ masks/boxes เอง (ข้ามรายละเอียดในโน้ตบุ๊กนี้)
    thumb = cv2.resize(img, (min(960, img.shape[1]), int(img.shape[0]*min(960, img.shape[1])/img.shape[1])))
    outp = OUT_DIR / f"thumb_{i+1}.jpg"
    cv2.imwrite(str(outp), thumb)
    thumb_paths.append(outp)

thumb_paths



ด้านล่างให้แทรกคำอธิบายแต่ละรูป (bounding boxes + segmentation masks overlay สำหรับแต่ละ instance)  
**ตัวอย่างคำอธิบาย (เขียนเองตามผลลัพธ์จริง):**
- รูปที่ 1: ตรวจพบจานข้าว 2 ใบ, ช้อน 1 คัน — บริเวณขอบจานมี mask ครอบคลุมครบถ้วน IoU สูง
- รูปที่ 2: ช้อนส้อมซ้อนกัน 2 ชิ้น — box ครอบคลุมแต่ mask ของส้อมรั่วหลุดเล็กน้อยบริเวณปลาย
- รูปที่ 3: จานแกง 1 ใบ + ช้อน 1 คัน — มี occlusion บางส่วนแต่ยังตรวจจับได้
- รูปที่ 4: จาน 1 ใบ — ค่าความเชื่อมั่นสูงกว่า 0.9


### (Optional) ส่ง PR Curve เข้าสู่ TensorBoard

In [ ]:

# ตัวอย่างการบันทึกรูป PR_curve.png เข้าสู่ TensorBoard เป็น image summary
# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter(log_dir="runs/tb_custom")
# import imageio, io
# for img_path in ["runs/val/exp/PR_curve.png", "runs/val/exp2/PR_curve.png"]:
#     p = Path(img_path)
#     if p.exists():
#         im = cv2.imread(str(p))
#         im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
#         writer.add_image(p.stem, im.transpose(2,0,1), 0)
# writer.flush()
# writer.close()
print("ตัวอย่างโค้ดสำหรับบันทึก PR_curve.png เข้า TensorBoard (ถูกคอมเมนต์ไว้)")



## 8) โครงสร้างชุดข้อมูล (YOLOv5 format)
ตัวอย่าง (det + seg ใช้โครงสร้างเดียวกัน แต่ label ของ seg จะเก็บ polygon ลงในไฟล์ `.txt` เช่นกัน)
```
dataset/
├─ images/
│  ├─ train/
│  └─ val/
└─ labels/
   ├─ train/
   └─ val/
```
- ในไฟล์ `data.yaml` ให้ระบุ path ของ `train` และ `val` (พาธไปยังโฟลเดอร์ `images`)
- รายชื่อคลาส (`names`) ต้องตรงกับ label

> ถ้าชุดข้อมูล segmentation ของคุณยังไม่ใช่รูปแบบ YOLOv5 (polygon ใน .txt) ให้แปลงก่อน



## 9) เช็กลิสต์ส่งงาน
- [ ] เทรน YOLOv5 (box) จาก `yolov5s.pt` พร้อมระบุ **epochs** และ **mini-batch** ที่แสดงการลู่เข้า
- [ ] เทรน YOLOv5-seg (seg) จาก `yolov5s-seg.pt` พร้อมระบุ **epochs** และ **mini-batch**
- [ ] แสดงกราฟ TensorBoard: **box regression-loss, objectness-loss, classification-loss, segmentation-loss, learning-rate, Accuracy (mAP@0.5), PR curve**
- [ ] รายงาน **mAP@0.5**
- [ ] โค้ดรวมความสามารถ **box + mask** เพื่อทำ **instance segmentation overlay**
- [ ] รูปตัวอย่าง inference อย่างน้อย **4 รูป** (overlay boxes + masks) พร้อมคำอธิบายแต่ละรูป
- [ ] ส่งเป็น `.ipynb` หรือ `.pdf` (จากเมนู File > Download as/Export)
